In [ ]:
# --- config ---
DATASET = "git_linux"            # transactions | senedd | git_linux | git_pytorch
MAX_ROWS = None               # random subsample to keep laptops quick; None for everything
MODEL_NAME = "all-MiniLM-L6-v2"

In [ ]:
# --- resolve preset + load ---
import random
from presets import PRESETS
from loaders import load_csv, render

preset = PRESETS[DATASET]
OUTPUT_DIR = f"../outputs/{DATASET}"

records = load_csv(preset["path"])
print(f"loaded {len(records)} rows from {preset['path']}")

if preset["filter"]:
    column, allowed = preset["filter"]
    records = [r for r in records if r.get(column) in allowed]
    print(f"kept {len(records)} rows where {column} is in {sorted(allowed)}")

if MAX_ROWS and len(records) > MAX_ROWS:
    records = random.Random(42).sample(records, MAX_ROWS)
    print(f"sampled {MAX_ROWS} rows (set MAX_ROWS = None for all)")

In [ ]:
# --- embed ---
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(MODEL_NAME)
texts = [render(r, preset["fields"]) for r in records]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

In [ ]:
# --- save ---
from pathlib import Path
import numpy as np
import json

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
np.save(out_dir / "embeddings.npy", embeddings)
with open(out_dir / "metadata.json", "w") as f:
    json.dump(records, f)
manifest = {
    "model_name": MODEL_NAME, "modality": "text",
    "n_items": len(embeddings), "dim": int(embeddings.shape[1]),
}
with open(out_dir / "manifest.json", "w") as f:
    json.dump(manifest, f)

print(f"saved {len(embeddings)} vectors to {out_dir}")